# Giskard Checks & Scan — Installation Guide

This notebook is a step-by-step walkthrough for installing and verifying **`giskard-checks`** (evals/scenario API for testing agentic systems) and **`giskard-scan`** (the agent vulnerability scanner), then running your first check and your first scan.

**Prerequisites**
- Python 3.12 or newer
- An API key for at least one LLM provider (OpenAI, Anthropic, Google, or Azure) — needed for LLM-as-judge checks and attack generation

Run the cells in order, top to bottom.

## Step 1 — Install the packages

```
pip install giskard-checks giskard-scan
```

The cell below runs the same command from inside the notebook using `%pip`, which installs into the kernel's current environment (works the same whether that's a plain venv, conda env, or Colab).

> **Contributing to giskard-oss instead?** If you cloned the [giskard-oss repo](https://github.com/Giskard-AI/giskard-oss) and want an editable install of the workspace source rather than the published packages, run `uv sync` from the repo root instead of the cell below.

In [ ]:
%pip install -q giskard-checks giskard-scan

## Step 2 — Verify the installation

Import both packages and print their versions. If this cell raises an `ImportError`, re-run Step 1 and restart the kernel.

In [ ]:
import giskard.checks
import giskard.scan

print(f"giskard-checks {giskard.checks.__version__}")
print(f"giskard-scan   {giskard.scan.__version__}")

## Step 3 — Configure an LLM provider

`giskard-checks` needs an LLM for LLM-as-judge checks (e.g. `Groundedness`), and `giskard-scan`'s attack generators need one too. Set the API key for **one** provider as an environment variable before running any LLM-backed check:

| Provider | Env var | Model string prefix |
|---|---|---|
| OpenAI | `OPENAI_API_KEY` | `openai/...` |
| Anthropic | `ANTHROPIC_API_KEY` | `anthropic/...` |
| Google (Gemini) | `GEMINI_API_KEY` (or `GOOGLE_API_KEY`) | `google/...` |
| Azure OpenAI | `AZURE_API_KEY`, `AZURE_API_BASE` | `azure/...` |

By default, `giskard-checks` uses `openai/gpt-4o-mini` for any check that doesn't specify its own model. Override this with the `GISKARD_CHECKS_DEFAULT_MODEL` environment variable, or in Python via `giskard.checks.set_default_generator(...)`.

Edit the cell below with your own key, then run it.

In [ ]:
import os

# Replace with your own key. This example uses OpenAI; swap in the env var
# from the table above if you're using a different provider.
os.environ["OPENAI_API_KEY"] = "sk-..."

# Optional: use a different default model for checks that don't specify one.
# os.environ["GISKARD_CHECKS_DEFAULT_MODEL"] = "anthropic/claude-sonnet-4-5"

## Step 4 — Run your first check (no LLM required)

A `Scenario` describes one interaction with your agent (the **target**) and one or more `Check`s that validate the result. `Equals` is a plain deterministic check, so this cell runs even before you've configured an LLM provider — a good way to confirm the core API works.

In [ ]:
from giskard.checks import Equals, Scenario


async def target(inputs: str) -> str:
    return f"Echo: {inputs}"


scenario = (
    Scenario("hello_world")
    .interact("Hello, Giskard!")
    .check(Equals(expected_value="Echo: Hello, Giskard!", key="trace.last.outputs"))
)

result = await scenario.run(target=target)
result.print_report()

## Step 5 — Run an LLM-powered check

`Groundedness` uses the LLM you configured in Step 3 to judge whether an answer is supported by the given context. This confirms your provider API key actually works end to end.

In [ ]:
from giskard.checks import Groundedness


async def target(inputs: str) -> str:
    return "Paris is the capital of France."


scenario = (
    Scenario("capital_of_france")
    .interact("What is the capital of France?")
    .check(
        Groundedness(
            context=["France's capital city is Paris. It is located in Europe."]
        )
    )
)

result = await scenario.run(target=target)
result.print_report()

## Step 6 — Run your first vulnerability scan

`giskard.scan.vulnerability_scan` generates and runs a suite of red-teaming scenarios (prompt injection, jailbreaks, adversarial attacks) against your target, using the same LLM configured in Step 3 for both attack generation and judging. `max_scenarios` keeps this first run small and fast; drop it to run the full suite.

In [ ]:
from giskard.scan import vulnerability_scan


async def target(inputs: str) -> str:
    return "I can help with orders, returns, and shipping questions."


scan_result = await vulnerability_scan(
    target=target,
    description="A customer support agent for an e-commerce company.",
    languages=["en"],
    max_scenarios=5,
)

## Step 7 — Export a report

`SuiteResult` (returned by both `scenario.run()` and `vulnerability_scan()`) can export to a self-contained static HTML report, or to JUnit XML for CI integration.

In [ ]:
scan_result.to_html("scan_report.html", title="My First Scan")
scan_result.to_junit_xml("scan_report.xml")

print(f"Pass rate: {scan_result.pass_rate:.1%}")
print("Reports written to scan_report.html and scan_report.xml")

## Next steps

- Browse the built-in checks: `from giskard.checks import ...` (see the [Giskard Checks docs](https://docs.giskard.ai/oss/checks))
- Explore scan customization options in `giskard.scan.types.ScanOptions`
- For a browser-based UI with live scan progress and separate Attacker/Target/Judge model configuration, see `apps/scan-ui/` in the [giskard-oss repo](https://github.com/Giskard-AI/giskard-oss)